# Análise Pipeline ABNT - v3.1

Notebook de analise, visualizacao e interpretacao dos resultados gerados pelo pipeline.

**Descricao fiel ao codigo atual:**
- este notebook consome os CSVs gerados pelo notebook de coleta SciELO/ArticleMeta
- a deteccao de secoes e estrutural: regex, fuzzy, spaCy em cabecalhos e spaCy em conteudo
- o BERTimbau entra na validacao semantica das secoes obrigatorias e na analise de coerencia semantica
- a validacao ABNT combina heading detectado e compatibilidade semantica por secao
- o pipeline tambem agrega analise lexical por secao, NER, citacoes, feedback heuristico e feedback opcional com LLM via Hugging Face
- OCR para PDFs escaneados e RAG para recuperacao contextual ficam como melhorias futuras desta versao


## Arquitetura tecnica do sistema - fiel ao codigo atual

1. **Coleta de dados**: o notebook `PLN_SciELO_API_3.ipynb` consulta a API ArticleMeta/SciELO, identifica revistas brasileiras e gera os CSVs base do sistema.
2. **Carga do corpus**: este notebook le `artigos_validos.csv` como entrada principal da analise.
3. **Pre-processamento**: o texto passa por filtro de idioma e normalizacao para as etapas seguintes.
4. **Deteccao estrutural de secoes**: headings e blocos sao identificados por `regex -> fuzzy -> spaCy cabecalho -> spaCy conteudo`.
5. **Validacao ABNT/NBR 6022**: as secoes obrigatorias sao avaliadas pela combinacao entre deteccao estrutural e validacao semantica com heuristica lexical + BERTimbau.
6. **Analises complementares**: o sistema calcula analise lexical por secao, coerencia semantica, NER spaCy e citacoes em padrao NBR 10520 simplificado.
7. **Saida e interpretacao**: os resultados sao exibidos em tabelas, graficos, feedback heuristico, feedback opcional com LLM via Hugging Face e persistencia em JSON + Parquet.

**Observacao tecnica:** o BERTimbau nao e o detector principal de headings; ele atua como validador semantico do conteudo detectado estruturalmente.
**Escopo desta versao:** OCR e RAG nao fazem parte do fluxo atual; ambos ficam registrados como melhorias futuras.


## 1. Objetivo do projeto

Avaliar artigos cientificos a partir dos CSVs gerados pela coleta SciELO/ArticleMeta, com foco em:
- deteccao estrutural de secoes do artigo
- validacao ABNT/NBR 6022 por combinacao entre heading detectado e validacao semantica
- analise lexical por secao (BoW para frequencia geral e TF-IDF para termos mais distintivos)
- coerencia semantica entre secoes com BERTimbau
- extracao de entidades com NER spaCy
- feedback interpretativo baseado em indicadores e, opcionalmente, em LLM via Hugging Face
- registro explicito de OCR e RAG como extensoes futuras, fora do escopo operacional atual


In [ ]:
# Setup do ambiente (Python 3.13): valida dependências e carrega o .env.
# A antiga lógica de downgrade de transformers/tokenizers foi removida de
# propósito: no Python 3.13 não existem wheels para as versões antigas e o
# pip tentava compilar tokenizers sem Build Tools, derrubando o kernel.
# A stack atual (transformers 5.x / tokenizers 0.22+) é a suportada.
import os
from pathlib import Path

_faltando = []
for _pacote in ("transformers", "sentence_transformers", "spacy", "torch"):
    try:
        __import__(_pacote)
    except ImportError:
        _faltando.append(_pacote)
if _faltando:
    raise RuntimeError(
        "Dependências ausentes no kernel: " + ", ".join(_faltando)
        + ". Instale com: pip install " + " ".join(_faltando)
    )

# .env do projeto (HUGGINGFACE_API_KEY -> HF_TOKEN): permite que o feedback
# via LLM (chamar_llm_analise_abnt) use a Hugging Face Inference API em
# qualquer célula; sem token, a função degrada para o fallback Ollama local.
try:
    from dotenv import load_dotenv
    _env = Path.cwd() / ".env"
    if _env.exists():
        load_dotenv(_env, override=False)
    _tok = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_API_KEY")
    if _tok and not os.environ.get("HF_TOKEN"):
        os.environ["HF_TOKEN"] = _tok
    print("HF_TOKEN disponível para a LLM:", "sim" if _tok else "não (fallback Ollama)")
except ImportError:
    print("python-dotenv não instalado — a LLM usará apenas variáveis já no ambiente.")

print("Ambiente OK: transformers, sentence-transformers, spaCy e torch importáveis.")


%run ./pipeline_abnt_funcoes_oficial.ipynb

import os

import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import seaborn as sns

from nltk.corpus import stopwords

import nltk



try:

    STOPWORDS_PT = set(stopwords.words("portuguese"))

except LookupError:

    nltk.download("stopwords", quiet=True)

    STOPWORDS_PT = set(stopwords.words("portuguese"))



STOPWORDS_PT.discard("não")

STOPWORDS_PT.discard("nao")



try:

    _sw_en = set(stopwords.words("english"))

    _sw_es = set(stopwords.words("spanish"))

except LookupError:

    nltk.download("stopwords", quiet=True)

    _sw_en = set(stopwords.words("english"))

    _sw_es = set(stopwords.words("spanish"))



STOPWORDS_LEXICO = STOPWORDS_PT | _sw_en | _sw_es

STOPWORDS_LEXICO.discard("não")

STOPWORDS_LEXICO.discard("nao")



nlp = carregar_modelo_spacy()

# BERTimbau-STS (substitui o BERTimbau cru -- ver comentario em MODEL_NAME_BERTIMBAU_STS
# no notebook de funcoes). tokenizer=None e convencao: model e um SentenceTransformer.
tokenizer, model = carregar_bertimbau_sts()

# FastText PT pre-treinado (NILC/fastText.cc) -- Jaccard semantico
# Ajuste o caminho abaixo para onde o arquivo .bin foi baixado.
CAMINHO_FASTTEXT_PT = "cc.pt.300.bin"
ft_model = carregar_fasttext_pt(CAMINHO_FASTTEXT_PT)

# KeyBERT -- reutiliza o modelo STS ja carregado (sem modelo extra)
kw_model = carregar_keybert(sts_model=model)

# NER LeNER-BR (PESSOA/ORGANIZACAO/LOCAL em PT)
ner_pipeline_lenerbr = carregar_ner_lenerbr()

# NER SciERC (Task/Method/Metric -- ingles, so para secao abstract)
ner_pipeline_scierc = carregar_ner_scierc()

# Camada 6: zero-shot NLI (USAR_ZERO_SHOT controla se carrega ou nao)
zs_pipeline = carregar_zero_shot(usar=USAR_ZERO_SHOT)


## 2. Carregamento do dataset gerado pela coleta SciELO/ArticleMeta

In [ ]:
# Entrada deste notebook: CSV gerado por PLN_SciELO_API_3.ipynb
base_dir_coleta_scielo = r"C:\Users\Maria Beatriz\Desktop\PLN projeto\PLN_BACKEND\PLN_SCIELO"
caminho_csv_artigos_validos = os.path.join(base_dir_coleta_scielo, "artigos_validos.csv")

df = pd.read_csv(caminho_csv_artigos_validos, encoding="utf-8-sig")
print(f"Dataset carregado: {df.shape[0]} linhas × {df.shape[1]} colunas")
print("Colunas:", list(df.columns))
display(df.head(3))


## 3. Filtro de idioma

In [ ]:
if 'df' not in globals():
    raise ValueError("DataFrame 'df' não encontrado. Execute primeiro a célula de carregamento.")

coluna_texto = 'texto' if 'texto' in df.columns else None
if coluna_texto is None:
    raise ValueError(f"Nenhuma coluna de texto encontrada. Colunas atuais: {list(df.columns)}")

df_original = df.copy()
df_original['indice_original'] = df_original.index

df_filtrado, df_pt, df_nao_pt = aplicar_filtro_idioma_percentual(
    df=df_original,
    coluna_texto=coluna_texto,
    limiar_pt=0.70,
    min_blocos_pt=2
)

print(f"Total de artigos: {len(df_filtrado)}")
print(f"Mantidos em PT:   {len(df_pt)}")
print(f"Não PT:           {len(df_nao_pt)}")

# Gráfico de distribuição
plt.figure(figsize=(6, 4))
vc = df_filtrado['manter_dataset_pt'].value_counts().rename(index={True: 'PT', False: 'Não PT'})
sns.barplot(x=vc.index, y=vc.values)
plt.title("Distribuição após filtro de idioma")
plt.xlabel("Classe")
plt.ylabel("Quantidade de artigos")
plt.tight_layout()
plt.show()

display(df_filtrado[['idioma_predominante','pct_pt','pct_en','pct_es','manter_dataset_pt']].head(10))


## 6. Validação ABNT/NBR 6022

`validar_estrutura_abnt` combina duas camadas complementares:

- **detecção estrutural de seções** por `regex -> fuzzy -> spaCy cabecalho -> spaCy conteudo`
- **validação semântica das seções obrigatórias** por heurística lexical + BERTimbau

Para cada seção obrigatória, o `score_semantico` é calculado por média ponderada `0.3 / 0.7` entre heurística e BERTimbau.
A classificação final depende da combinação entre **heading detectado** e **conteúdo semanticamente compatível**.


In [ ]:
amostra_sec = df_pt.dropna(subset=[coluna_texto]).sample(
    n=min(30, len(df_pt)),
    random_state=42
)

rows_sec        = []
secoes_por_artigo = {}

for i, (idx, row) in enumerate(amostra_sec.iterrows(), start=1):
    print(f"Processando seções {i}/{len(amostra_sec)} — índice {idx}")
    texto  = preparar_texto_para_estrutura(str(row[coluna_texto]))

    # Com BERTimbau para ativar validação híbrida completa
    # Se for lento no seu ambiente, troque tokenizer=None, model=None
    secoes = detectar_secoes(texto, nlp=None)
    secoes_por_artigo[idx] = secoes

    val = validar_estrutura_abnt(secoes, texto, tokenizer, model)
    res = val.get("_resumo", {})

    labels = [k for k in secoes if not k.startswith("_")]
    rows_sec.append({
        "indice_df":            idx,
        "n_secoes_detectadas":  len(labels),
        "secoes_detectadas":    ", ".join(sorted(labels)[:12]),
        "score_abnt":           res.get("score_conformidade"),
        "conforme":             res.get("conforme_nbr6022"),
        "n_criticos":           len(res.get("criticos", [])),
        "criticos":             ", ".join(res.get("criticos", [])),
        "n_avisos":             len(res.get("avisos", [])),
        "n_observacoes":        len(res.get("observacoes", [])),
    })

df_sec = pd.DataFrame(rows_sec)
display(df_sec.head(10))

# Histograma de seções detectadas
plt.figure(figsize=(8, 4))
sns.histplot(df_sec["n_secoes_detectadas"].dropna(), bins=12, kde=True)
plt.title("Distribuição do número de seções detectadas")
plt.xlabel("Número de seções detectadas")
plt.ylabel("Quantidade de artigos")
plt.tight_layout()
plt.show()


### 6.1 Detalhes híbridos por seção obrigatória

Tabela com score semântico e nível para cada seção obrigatória de um artigo exemplo.


In [ ]:
if not secoes_por_artigo:
    raise ValueError("Nenhum artigo processado na seção 6. Execute a célula anterior primeiro.")
idx_ex  = list(secoes_por_artigo.keys())[min(5, len(secoes_por_artigo) - 1)]
row_ex  = df_pt.loc[idx_ex]
txt_ex  = preparar_texto_para_estrutura(str(row_ex[coluna_texto]))
sec_ex  = secoes_por_artigo[idx_ex]

val_ex  = validar_estrutura_abnt(sec_ex, txt_ex, tokenizer, model)
det_hib = val_ex["_resumo"]["detalhes_hibrido"]

_cols_hibrido = [
    "titulo_presente", "score_lexical", "score_bert",
    "score_semantico", "threshold", "conteudo_presente", "nivel", "status", "mensagem"
]

df_hibrido_raw = pd.DataFrame(det_hib).T
_cols_hibrido_existentes = [c for c in _cols_hibrido if c in df_hibrido_raw.columns]
df_hibrido = df_hibrido_raw[_cols_hibrido_existentes]

def colorir_nivel(val):
    cores = {"ok":"#c7e9c0","observacao":"#9ecae1","aviso":"#fdae6b","critico":"#fc8d59"}
    return f"background-color: {cores.get(val, '')}"

def _fmt_decimal(valor, casas=3):
    if valor is None or pd.isna(valor):
        return "—"
    return f"{float(valor):.{casas}f}"

_fmt = {}
for _c in ["score_lexical", "score_bert", "score_semantico"]:
    if _c in _cols_hibrido_existentes:
        _fmt[_c] = lambda x: _fmt_decimal(x, 3)
if "threshold" in _cols_hibrido_existentes:
    _fmt["threshold"] = lambda x: _fmt_decimal(x, 2)

_style = df_hibrido.style
if "nivel" in _cols_hibrido_existentes:
    _style = _style.map(colorir_nivel, subset=["nivel"])
_style = _style.format(_fmt).set_caption(f"Validação semântica híbrida — artigo índice {idx_ex}")
display(_style)


## 7. Análise lexical com BoW e TF-IDF

> **v3:** `analisar_lexico` retorna `por_secao` — não há mais `artigo_completo`.  
> ngram fixado em (1,2). TF-IDF com `sublinear_tf=True` para raros; BoW para frequência geral.


In [ ]:
lex_ex = analisar_lexico(
    txt_ex,
    sec_ex,
    stopwords_pt=STOPWORDS_LEXICO,
    top_n=20,
)

print(f"ngram usado: {lex_ex.get('ngram_range_usado')}")
print(f"Seções com léxico: {list(lex_ex['por_secao'].keys())}")

# Agrega top termos do artigo completo a partir de por_secao
from collections import Counter

bow_agregado   = Counter()
tfidf_agregado = {}

for label, dados in lex_ex["por_secao"].items():
    for termo, freq in dados["bow"].items():
        bow_agregado[termo] += freq
    for termo, score in dados["tfidf"]:
        if termo not in tfidf_agregado or score > tfidf_agregado[termo]:
            tfidf_agregado[termo] = score

df_top_bow = pd.DataFrame(bow_agregado.most_common(20), columns=["termo","frequencia"])
df_top_tfidf = (
    pd.DataFrame(sorted(tfidf_agregado.items(), key=lambda x: -x[1])[:20],
                 columns=["termo","score"])
)

display(df_top_bow)
display(df_top_tfidf)

if df_top_bow.empty and df_top_tfidf.empty:
    print("⚠️  Nenhum termo disponível para visualização lexical neste artigo.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    if not df_top_bow.empty:
        sns.barplot(data=df_top_bow, y="termo", x="frequencia", ax=axes[0])
        axes[0].set_title("Top 20 termos BoW (frequência geral)")
        axes[0].set_xlabel("Frequência")
    else:
        axes[0].text(0.5, 0.5, "Sem dados BoW", ha="center", va="center")
        axes[0].set_axis_off()

    if not df_top_tfidf.empty:
        sns.barplot(data=df_top_tfidf, y="termo", x="score", ax=axes[1])
        axes[1].set_title("Top 20 termos TF-IDF (termos raros/temáticos)")
        axes[1].set_xlabel("Score TF-IDF")
    else:
        axes[1].text(0.5, 0.5, "Sem dados TF-IDF", ha="center", va="center")
        axes[1].set_axis_off()

    plt.tight_layout()
    plt.show()


### 7.1 Top termos por seção

Comparativo BoW × TF-IDF por seção detectada.


In [ ]:
secoes_disponiveis = list(lex_ex["por_secao"].keys())
print("Seções disponíveis:", secoes_disponiveis)

# Plota as 4 primeiras seções
for label in secoes_disponiveis[:4]:
    dados = lex_ex["por_secao"][label]
    df_bow_s   = pd.DataFrame(list(dados["bow"].items())[:15],  columns=["termo","frequencia"])
    df_tfidf_s = pd.DataFrame(dados["tfidf"][:15], columns=["termo","score"])

    if df_bow_s.empty and df_tfidf_s.empty:
        print(f"⚠️  Sem dados lexicais suficientes para a seção '{label}'.")
        continue

    fig, axes = plt.subplots(1, 2, figsize=(14, 3))
    if not df_bow_s.empty:
        sns.barplot(data=df_bow_s, y="termo", x="frequencia", ax=axes[0], color="#4878d0")
        axes[0].set_title(f"BoW — {label}")
    else:
        axes[0].text(0.5, 0.5, "Sem dados BoW", ha="center", va="center")
        axes[0].set_axis_off()
    if not df_tfidf_s.empty:
        sns.barplot(data=df_tfidf_s, y="termo", x="score", ax=axes[1], color="#ee854a")
        axes[1].set_title(f"TF-IDF — {label}")
    else:
        axes[1].text(0.5, 0.5, "Sem dados TF-IDF", ha="center", va="center")
        axes[1].set_axis_off()
    plt.tight_layout()
    plt.show()


## 9. Coerência semântica entre seções com BERTimbau

Similaridade cosseno entre pares de seções detectadas, usada como medida de coerência global do artigo.


In [ ]:
if model is None:
    print("⚠️  BERTimbau não foi carregado. Pulando análise semântica.")
else:
    try:
        sem_ex = analisar_coerencia_semantica(txt_ex, sec_ex, tokenizer, model, threshold=0.50)
        if sem_ex.get("erro"):
            print(f"❌ Erro: {sem_ex['erro']}")
        else:
            labels_sem = sem_ex["labels"]
            M = np.array(sem_ex["matriz"])
            plt.figure(figsize=(7, 6))
            sns.heatmap(M, xticklabels=labels_sem, yticklabels=labels_sem,
                        cmap="YlOrRd", annot=True, fmt=".2f")
            plt.title("Heatmap semantico BERTimbau")
            plt.xticks(rotation=45, ha="right")
            plt.tight_layout()
            plt.show()
    except Exception as e:
        print(f"❌ Erro ao executar análise semântica: {e}")


In [ ]:
# Tabela de correlação semântica em %
if 'sem_ex' in dir() and not sem_ex.get("erro") and len(sem_ex.get("labels", [])) > 0:
    labels_tab = sem_ex["labels"]
    M_tab = np.array(sem_ex["matriz"])

    df_correlacao = pd.DataFrame(
        (M_tab * 100).round(1),
        index=labels_tab, columns=labels_tab
    )

    def colorir_correlacao(val):
        if val >= 80:   return "background-color: #1b7837; color: white; font-weight: bold; text-align: center"
        elif val >= 60: return "background-color: #74c476; color: black; font-weight: bold; text-align: center"
        elif val >= 40: return "background-color: #fdae6b; color: black; font-weight: bold; text-align: center"
        else:           return "background-color: #d73027; color: white; font-weight: bold; text-align: center"

    def realcar_diagonal(df):
        estilos = pd.DataFrame("", index=df.index, columns=df.columns)
        for i in range(min(df.shape)):
            estilos.iloc[i, i] = "background-color: #aaaaaa; color: white; font-weight: bold; text-align: center"
        return estilos

    styled = (
        df_correlacao.style
        .map(colorir_correlacao)
        .apply(realcar_diagonal, axis=None)
        .set_caption("Correlação semântica entre seções (%) — BERTimbau")
        .format("{:.1f}%")
    )
    display(styled)

    np.fill_diagonal(M_tab, np.nan)
    _M_flat = M_tab.flatten()
    _validos = _M_flat[~np.isnan(_M_flat)]

    if len(labels_tab) < 2 or len(_validos) == 0:
        print("⚠️  Apenas uma seção detectada — não é possível calcular pares de coerência.")
    else:
        idx_max = np.unravel_index(np.nanargmax(M_tab), M_tab.shape)
        idx_min = np.unravel_index(np.nanargmin(M_tab), M_tab.shape)
        print(f"📈 Par mais coerente:   {labels_tab[idx_max[0]]} ↔ {labels_tab[idx_max[1]]}  ({M_tab[idx_max]*100:.1f}%)")
        print(f"📉 Par menos coerente: {labels_tab[idx_min[0]]} ↔ {labels_tab[idx_min[1]]}  ({M_tab[idx_min]*100:.1f}%)")
else:
    print("⚠️  Análise semântica não disponível. Execute a célula anterior primeiro.")


## 10. NER

In [ ]:
texto_ner_ex = preparar_texto_para_ner(str(row_ex[coluna_texto]))
ner_ex = extrair_entidades(texto_ner_ex, nlp)

if "erro" in ner_ex:
    print(f"❌ {ner_ex['erro']}")
else:
    for tipo, entidades in ner_ex.items():
        print(f"  {tipo}: {entidades[:8]}")


## 11. Agregação dos indicadores

> **v3:** `executar_pipeline_dataset` passa `tokenizer` e `model` para ativar a validação  
> híbrida no lote. Para pular BERTimbau no lote (mais rápido), troque por `tokenizer=None, model=None`.


In [ ]:
df_resultados_pipeline = executar_pipeline_dataset(
    df_pt=df_pt,
    coluna_texto=coluna_texto,
    nlp=None,
    tokenizer=tokenizer,          # None = SentenceTransformer (BERTimbau-STS)
    model=model,
    stopwords_pt=STOPWORDS_LEXICO,
    n_amostras=min(5, len(df_pt)),
    random_state=42,
    ft_model=ft_model,            # Jaccard semantico (FastText PT)
    kw_model=kw_model,            # termos-chave por similaridade (KeyBERT)
    ner_pipeline_lenerbr=ner_pipeline_lenerbr,  # NER LeNER-BR
    ner_pipeline_scierc=ner_pipeline_scierc,    # NER SciERC (so abstract)
    zs_pipeline=zs_pipeline,                    # camada 6: zero-shot NLI
)

# Colunas principais — estrutura + citações NBR 10520:2023 + novos indicadores
cols_base = [
    "indice_df", "score_abnt_heuristico", "conforme_nbr6022",
    "n_criticos", "n_avisos", "n_observacoes",
    "score_semantico_hibrido_medio", "n_secoes_detectadas",
    "ordem_secoes_correta",          # verificacao de ordem das secoes (NBR 6022)
    "jaccard_semantico_medio",       # Jaccard via FastText (aceita sinonimos)
    "n_termos_cientificos_abstract", # NER SciERC no abstract
    "n_entidades_data",              # datas extraidas por regex
]
cols_cit = [c for c in ["citacoes_diretas","citacoes_indiretas","citacoes_apud",
                         "score_citacoes","status_citacoes"]
            if c in df_resultados_pipeline.columns]

display(df_resultados_pipeline[[c for c in cols_base + cols_cit
                                  if c in df_resultados_pipeline.columns]].head(10))


In [ ]:
# Resumo de ausências por seção
faltas = []
for _, row in df_resultados_pipeline.iterrows():
    for col in ["criticos_lista","avisos_lista"]:
        val = row.get(col,"[]")
        try:
            import ast
            items = ast.literal_eval(val) if isinstance(val, str) else val
        except Exception:
            items = [x.strip() for x in str(val).strip("[]").split(",") if x.strip()]
        faltas.extend(items)

df_faltas = pd.Series(faltas).value_counts().reset_index()
df_faltas.columns = ["secao","qtd_ausencias"]
display(df_faltas.head(15))

if not df_faltas.empty:
    plt.figure(figsize=(9, 4))
    sns.barplot(data=df_faltas.head(15), x="secao", y="qtd_ausencias")
    plt.title("Seções mais problemáticas (críticos + avisos)")
    plt.xlabel("Seção"); plt.ylabel("Ocorrências")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

# Gráfico barras agrupadas + pizza
df_plot = df_resultados_pipeline.copy()
df_plot["artigo"] = ["Artigo " + str(i+1) for i in range(len(df_plot))]
colunas = ["n_criticos","n_avisos","n_observacoes"]
cores   = {"n_criticos":"#d62728","n_avisos":"#ff7f0e","n_observacoes":"#1f77b4"}
labels_legenda = {"n_criticos":"Críticos","n_avisos":"Avisos","n_observacoes":"Observações"}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
largura = 0.25
x = np.arange(len(df_plot))
total_geral = df_plot[colunas].values.sum()

for i, col in enumerate(colunas):
    valores = df_plot[col].values
    barras  = axes[0].bar(x + i*largura, valores, width=largura,
                          label=labels_legenda[col], color=cores[col])
    for barra, val in zip(barras, valores):
        pct = (val/total_geral*100) if total_geral > 0 else 0
        if val > 0:
            axes[0].text(barra.get_x()+barra.get_width()/2,
                         barra.get_height()+0.05, f"{pct:.1f}%",
                         ha="center", va="bottom", fontsize=7.5)

axes[0].set_title("Violações por artigo (NBR 6022)")
axes[0].set_xticks(x + largura)
axes[0].set_xticklabels(df_plot["artigo"], rotation=15)
axes[0].legend()

totais = [df_plot[col].sum() for col in colunas]
axes[1].pie(totais, labels=["Críticos","Avisos","Observações"],
            colors=["#d62728","#ff7f0e","#1f77b4"], autopct="%1.1f%%", startangle=90)
axes[1].set_title("Proporção geral de violações")
plt.tight_layout()
plt.show()


### 11.1 Inspeção manual de artigo específico

> **v3:** exibe detalhes híbridos por seção obrigatória.


In [ ]:
# Troca o número abaixo (0, 1, 2, 3, 4) para inspecionar outro artigo
if df_resultados_pipeline.empty:
    raise ValueError("df_resultados_pipeline está vazio. Execute a célula da seção 11 primeiro.")
i = min(2, len(df_resultados_pipeline) - 1)

idx     = int(df_resultados_pipeline.iloc[i]["indice_df"])
row     = df_pt.loc[idx]
resultado = analisar_artigo(
    str(row[coluna_texto]),
    nlp=nlp,
    tokenizer=tokenizer,
    model=model,
    stopwords_pt=STOPWORDS_LEXICO,
    ft_model=ft_model,
    kw_model=kw_model,
    ner_pipeline_lenerbr=ner_pipeline_lenerbr,
    ner_pipeline_scierc=ner_pipeline_scierc,
    zs_pipeline=zs_pipeline,
)

print("=" * 55)
print(f"ARTIGO {i+1} — índice {idx}")
print("=" * 55)
if resultado.get("_erro"):
    print(f"⚠️  Pipeline executado com observação: {resultado['_erro']}")

# _validacao contém a saída bruta de validar_estrutura_abnt
res = resultado["_validacao"]["_resumo"]
print("\n📋 SEÇÕES DETECTADAS:")
print(list(resultado["secoes"].keys()))
print("\n🔴 CRÍTICOS:", res.get("criticos") or "Nenhum")
print("AVISOS:     ", res.get("avisos") or "Nenhum")
print("OBSERV.:    ", res.get("observacoes") or "Nenhuma")
print(f"\n📊 SCORE: {res.get('score_conformidade')}   ✅ CONFORME: {res.get('conforme_nbr6022')}")

# Tabela híbrida por seção obrigatória
det = res.get("detalhes_hibrido", {})
if det:
    print("\n🧠 VALIDAÇÃO SEMÂNTICA HÍBRIDA por seção obrigatória:")
    _cols_det = [
        "titulo_presente", "score_lexical", "score_bert",
        "score_semantico", "conteudo_presente", "nivel", "status", "mensagem"
    ]
    df_det_raw = pd.DataFrame(det).T
    _cols_det_existentes = [c for c in _cols_det if c in df_det_raw.columns]
    df_det = df_det_raw[_cols_det_existentes]
    display(df_det.style.map(
        lambda v: {"ok":"background-color:#c7e9c0","observacao":"background-color:#9ecae1",
                   "aviso":"background-color:#fdae6b","critico":"background-color:#fc8d59"}.get(v, ""),
        subset=["nivel"] if "nivel" in _cols_det_existentes else []
    ))

# Tabela de seções com indicadores lexicais (Bloco 2)
print("\n📖 INDICADORES LEXICAIS POR SEÇÃO:")
df_lex = pd.DataFrame({
    label: {
        "status":              sec.get("status"),
        "tem_lexico":          sec.get("tem_lexico"),
        "jaccard_score":       sec.get("jaccard_score"),
        "jaccard_semantico":   sec.get("jaccard_semantico"),   # FastText
        "indicador_lexical":   sec.get("indicador_lexical"),
        "keybert_top_terms":   ", ".join(sec.get("keybert_top_terms", [])[:5]),
        "score_lexical":       sec.get("score_lexical"),
        "score_final":         sec.get("score_final"),
    }
    for label, sec in resultado["secoes"].items()
}).T
display(df_lex)

# Ordem das seções (verificacao NBR 6022)
_ordem = resultado.get("_validacao", {}).get("_resumo", {}).get("ordem_secoes", {})
if _ordem:
    _ok = _ordem.get("ordem_correta")
    print(f"\n📋 ORDEM DAS SEÇÕES: {'✅ Correta' if _ok else '⚠️  Fora de ordem'}")
    if not _ok:
        for par in _ordem.get("pares_fora_de_ordem", []):
            print(f"  → '{par[1]}' aparece antes de '{par[0]}'")

# NER — LeNER-BR + SciERC (abstract)
_ner = resultado.get("ner", {})
_ent_abst = _ner.get("entidades_cientificas_abstract", {})
if _ent_abst:
    print("\n🔬 TERMOS CIENTÍFICOS DO ABSTRACT (SciERC):")
    for cat, termos in _ent_abst.items():
        print(f"  {cat}: {', '.join(termos[:5])}")


In [ ]:
# Texto das seções obrigatórias suspeitas
# _secoes_raw: posições inteiras (para _extrair_texto_secao)
# resultado["secoes"]: dicts de validação por seção
secoes_validas = {k: v for k, v in resultado["_secoes_raw"].items()
                  if not k.startswith("_") and isinstance(v, int)}

for label in ["introducao", "conclusao"]:
    if label not in secoes_validas:
        print(f"\n⚠️  Seção '{label}' não detectada.")
        continue
    print(f"\n{'='*55}\nTEXTO DA SEÇÃO: {label.upper()}\n{'='*55}")
    txt = _extrair_texto_secao(resultado["_texto_estruturado"], secoes_validas, label)
    print(txt[:2500])



## 12. Citações — NBR 10520 (simplificado)

> `avaliar_citacoes_simples` avalia apenas **citações diretas** e **citações indiretas**.
>
> - **Diretas**: texto entre aspas com chamada autor-data — identifica com e sem indicação de página
> - **Indiretas**: chamada autor-data sem aspas (paráfrase) — parentéticas, textuais e introdutórias
>
> O resultado já está disponível em `resultado["citacoes"]` (calculado dentro de `analisar_artigo`).
>
> **Fora do escopo nesta fase:** apud, validação citação-referência, NER, análise semântica de citações.


In [ ]:
# Inspeção de conformidade de citações do artigo já analisado em 11.1
# resultado["citacoes"] é gerado por avaliar_conformidade_citacoes_nbr10520() dentro de analisar_artigo()
cit = resultado.get("citacoes", {})
_score_cit = cit.get("score_citacoes")
_score_cit_fmt = f"{float(_score_cit):.2f}" if _score_cit is not None else "n/d"

print(f"Total de citações detectadas : {cit.get('total', 0)}")
print(f"Status                       : {cit.get('status', 'n/d')}")
print(f"Score de citações            : {_score_cit_fmt}")
print(f"\nDiretas  : {cit.get('diretas', {}).get('count', 0)}"
      f"  (com página: {cit.get('diretas', {}).get('com_pagina', 0)} | sem página: {cit.get('diretas', {}).get('sem_pagina', 0)})")
print(f"Indiretas: {cit.get('indiretas', {}).get('count', 0)}")

if cit.get("alertas"):
    print("\n⚠️  Alertas de conformidade de citações (NBR 10520):")
    for a in cit.get("alertas", []):
        print(f"   • {a}")
else:
    print("\n✅ Nenhum alerta de conformidade.")

df_cit_resumo = pd.DataFrame([{
    "tipo": "Diretas",
    "total": cit.get("diretas", {}).get("count", 0),
    "com_pagina": cit.get("diretas", {}).get("com_pagina", 0),
    "sem_pagina": cit.get("diretas", {}).get("sem_pagina", 0),
}, {
    "tipo": "Indiretas",
    "total": cit.get("indiretas", {}).get("count", 0),
    "com_pagina": "-",
    "sem_pagina": "-",
}])
display(df_cit_resumo)


In [ ]:
# Distribuição da conformidade de citações na amostra processada pelo pipeline
cols_cit = ["citacoes_diretas", "citacoes_indiretas", "score_citacoes", "status_citacoes"]
if all(c in df_resultados_pipeline.columns for c in cols_cit):
    df_cit = df_resultados_pipeline[["indice_df"] + cols_cit].copy()
    df_cit["artigo"] = ["Artigo " + str(i + 1) for i in range(len(df_cit))]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    x = np.arange(len(df_cit))
    w = 0.35
    axes[0].bar(x - w / 2, df_cit["citacoes_diretas"], width=w, label="Diretas", color="#1f77b4")
    axes[0].bar(x + w / 2, df_cit["citacoes_indiretas"], width=w, label="Indiretas", color="#ff7f0e")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(df_cit["artigo"], rotation=15)
    axes[0].set_title("Citações detectadas por artigo (NBR 10520)")
    axes[0].set_ylabel("Quantidade")
    axes[0].legend()

    _cores_status = {
        "adequado": "#2ca02c",
        "requer revisão": "#ff7f0e",
        "não identificado": "#d62728",
    }
    cores = df_cit["status_citacoes"].map(_cores_status).fillna("#aec7e8")
    axes[1].bar(df_cit["artigo"], df_cit["score_citacoes"], color=cores)
    axes[1].set_title("Conformidade de citações por artigo")
    axes[1].set_ylabel("Score (0–1)")
    axes[1].set_ylim(0, 1.1)
    axes[1].axhline(0.8, color="green", linestyle="--", alpha=0.5, label="Adequado ≥ 0.8")
    axes[1].axhline(0.5, color="orange", linestyle="--", alpha=0.5, label="Parcial ≥ 0.5")
    axes[1].legend(fontsize=8)
    plt.xticks(rotation=15)

    plt.tight_layout()
    plt.show()

    display(df_cit.drop(columns=["artigo"]))
else:
    print("⚠️  Colunas de conformidade de citações não encontradas. Execute a célula de pipeline antes.")



### 12.1 Saída estruturada do pipeline (`gerar_saida_analise`)

> `gerar_saida_analise(resultado)` converte a saída de `analisar_artigo` para um formato estruturado:
> `score_geral`, `status_geral`, `resumo_resultados`, `estrutura_abnt`, `secoes` (com indicadores lexicais), `citacoes`.
>
> Esta saída serve para análise no notebook, salvamento em banco de dados e futura integração com Streamlit.
> O alias `gerar_saida_frontend` aponta para a mesma função.


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Visualização estruturada — consome gerar_saida_analise(resultado)
# ──────────────────────────────────────────────────────────────────────────────

# Verificações defensivas de variáveis globais
_variaveis_ok = True
for _var, _nome in [
    ("df_pt",                  "df_pt"),
    ("df_resultados_pipeline", "df_resultados_pipeline"),
    ("coluna_texto",           "coluna_texto"),
    ("STOPWORDS_LEXICO",       "STOPWORDS_LEXICO"),
]:
    if _var not in globals() or globals()[_var] is None:
        print(f"⚠️  Variável '{_nome}' não encontrada ou vazia. "
              f"Execute as células anteriores antes de continuar.")
        _variaveis_ok = False

if not _variaveis_ok:
    raise SystemExit("Abortando visualização: variáveis ausentes acima.")

if 'model' not in globals() or model is None:
    print("⚠️  BERTimbau indisponível nesta sessão. A visualização continuará com fallback parcial.")
if 'nlp' not in globals() or nlp is None:
    print("⚠️  spaCy indisponível nesta sessão. A visualização continuará com fallback parcial.")
if len(df_pt) == 0 or len(df_resultados_pipeline) == 0:
    raise SystemExit("Abortando visualização: df_pt ou df_resultados_pipeline estão vazios.")

if "resultado" not in globals():
    print("⚠️  'resultado' não definido. Executando analisar_artigo no primeiro artigo disponível...")
    _idx_vis = int(df_pt.index[0])
    resultado = analisar_artigo(
        str(df_pt.loc[_idx_vis, coluna_texto]),
        nlp=nlp, tokenizer=tokenizer, model=model,
        stopwords_pt=STOPWORDS_LEXICO,
        ft_model=ft_model, kw_model=kw_model,
        ner_pipeline_lenerbr=ner_pipeline_lenerbr,
        ner_pipeline_scierc=ner_pipeline_scierc,
        zs_pipeline=zs_pipeline,
    )

saida = gerar_saida_analise(resultado)
_erro_saida = resultado.get("_erro") or saida.get("metricas", {}).get("erro_pipeline")
_secoes_saida = saida.get("secoes", {}) or {}
_secoes_vis = {}
for _label, _sec in _secoes_saida.items():
    _status_atual = _sec.get("status", "?")
    _presente = _sec.get("presente", _status_atual != "Não contém seção")
    _secoes_vis[_label] = {
        "presente": _presente,
        **_sec,
    }

from IPython.display import Markdown, display
import math

def _fmt_num(_valor, casas=3):
    if _valor is None:
        return "n/d"
    try:
        _num = float(_valor)
        if math.isnan(_num):
            return "n/d"
        return f"{_num:.{casas}f}"
    except Exception:
        return str(_valor)

# ── 1. Diagnóstico geral ──────────────────────────────────────────────────────
print("=" * 60)
print("DIAGNÓSTICO GERAL")
print("=" * 60)
if _erro_saida:
    print(f"⚠️  Observação do pipeline: {_erro_saida}")
print(f"  Score geral  : {saida['score_geral']}/100")
print(f"  Status geral : {saida['status_geral']}")
print(f"  Idioma       : {saida['idioma']} ({saida['status_idioma']})")
_res = saida["resumo_resultados"]
print(f"  Aprovados    : {_res['aprovados']}")
print(f"  Observações  : {_res['observacoes']}")
print(f"  Reprovados   : {_res['reprovados']}")
_qtd = sum(1 for _sec in _secoes_vis.values() if _sec.get("presente"))
print(f"  Seções obrigatórias presentes: {_qtd}/{len(_secoes_vis) or 5}")

# ── 2. Seções obrigatórias ────────────────────────────────────────────────────
print("\n── Seções obrigatórias ──────────────────────────────────────────────────────")
_COR_STATUS = {
    "Contém seção":               "background-color:#c7e9c0",
    "Contém seção com observação":"background-color:#9ecae1",
    "Requer revisão":             "background-color:#fdae6b",
    "Não contém seção":           "background-color:#fc8d59",
}
_EMOJI = {
    "Contém seção":               "✅",
    "Contém seção com observação":"🔵",
    "Requer revisão":             "⚠️",
    "Não contém seção":           "🔴",
}

for _label, _sec in _secoes_vis.items():
    _emoji  = _EMOJI.get(_sec.get("status", ""), "❓")
    _pres   = "presente" if _sec.get("presente") else "ausente"
    _status = _sec.get("status", "?")
    _sc_sem = _sec.get("score_semantico")
    _sc_lex = _sec.get("score_lexical", 0)
    _jac    = _sec.get("jaccard_score")
    if _sc_sem is not None and _jac is not None and not pd.isna(_sc_sem) and not pd.isna(_jac):
        print(
            f"  {_emoji} {_label:<15} | {_pres:<8} | {_status:<35}"
            f"| sem={_sc_sem:.3f} | lex={_sc_lex:.3f} | jaccard={_jac:.3f}"
        )
    else:
        print(f"  {_emoji} {_label:<15} | {_pres:<8} | {_status}")

# Tabela estilizada das seções obrigatórias
if _secoes_vis:
    _colunas_sec = [
        "presente", "status", "tem_lexico", "tem_semantico",
        "score_lexical", "score_semantico", "jaccard_score",
        "indicador_lexical", "titulo_detectado",
    ]
    _primeira_secao = next(iter(_secoes_vis.values()))
    _colunas_existentes = [c for c in _colunas_sec if c in _primeira_secao]
    df_sec_vis = pd.DataFrame(_secoes_vis).T[_colunas_existentes]

    _fmt_sec = {}
    for _c in ["score_lexical", "score_semantico", "jaccard_score"]:
        if _c in _colunas_existentes:
            _fmt_sec[_c] = lambda x: "—" if x is None or pd.isna(x) else f"{x:.3f}"

    _style = df_sec_vis.style
    if "status" in _colunas_existentes:
        _style = _style.map(lambda v: _COR_STATUS.get(v, ""), subset=["status"])
    display(
        _style
        .format(_fmt_sec)
        .set_caption("Seções obrigatórias — presente / score semântico / lexical / Jaccard")
    )
else:
    print("  Nenhuma seção obrigatória disponível na saída estruturada.")

# ── 3. BoW, TF-IDF e KeyBERT top terms por seção ────────────────────────────
print("\n── Top terms por seção (BoW | TF-IDF | KeyBERT) ────────────────────────────")
for _label, _sec in _secoes_vis.items():
    _bow    = _sec.get("bow_top_terms", [])
    _tfidf  = _sec.get("tfidf_top_terms", [])
    _keyb   = _sec.get("keybert_top_terms", [])
    if _bow or _tfidf or _keyb:
        print(f"\n  [{_label}]")
        print(f"    BoW    : {', '.join(_bow[:8]) or '—'}")
        print(f"    TF-IDF : {', '.join(_tfidf[:8]) or '—'}")
        print(f"    KeyBERT: {', '.join(_keyb[:8]) or '—'}  ← por similaridade semântica")

# ── 3.1 Termos científicos do abstract (SciERC) ───────────────────────────────
_ent_abst = resultado.get("ner", {}).get("entidades_cientificas_abstract", {})
if _ent_abst:
    print("\n── Termos científicos do abstract (SciERC) ──────────────────────────────────")
    for _cat, _termos in _ent_abst.items():
        print(f"  {_cat}: {', '.join(_termos)}")

# ── 3.2 Ordem das seções (NBR 6022) ──────────────────────────────────────────
_ordem_res = resultado.get("_validacao", {}).get("_resumo", {}).get("ordem_secoes", {})
if _ordem_res:
    _ok_ordem = _ordem_res.get("ordem_correta")
    print(f"\n── Ordem das seções: {'✅ Correta' if _ok_ordem else '⚠️  Fora de ordem (ver pares abaixo)'}")
    for _par in _ordem_res.get("pares_fora_de_ordem", []):
        print(f"  → '{_par[1]}' aparece antes de '{_par[0]}'")

# ── 4. Citações ───────────────────────────────────────────────────────────────
print("\n── Citações (NBR 10520) ─────────────────────────────────────────────────────")
_cit = saida.get("citacoes", {})
print(f"  Status      : {_cit.get('status', 'n/d')}  |  Score: {_fmt_num(_cit.get('score_citacoes'), 2)}")
print(f"  Diretas     : {_cit.get('diretas', {}).get('count', 0)}"
      f"  (com pág: {_cit.get('diretas', {}).get('com_pagina', 0)} | sem pág: {_cit.get('diretas', {}).get('sem_pagina', 0)})")
print(f"  Indiretas   : {_cit.get('indiretas', {}).get('count', 0)}")
for _alerta in _cit.get("alertas", []):
    print(f"  ⚠️  {_alerta}")

# ── 5. NER ────────────────────────────────────────────────────────────────────
print("\n── NER ──────────────────────────────────────────────────────────────────────")
_ner = saida.get("ner", {})
print(f"  Autores detectados : {_ner.get('autores_detectados', [])[:5]}")
print(f"  Organizações       : {_ner.get('organizacoes_detectadas', [])[:5]}")

# ── 6. Métricas ───────────────────────────────────────────────────────────────
print("\n── Métricas ─────────────────────────────────────────────────────────────────")
_met = saida.get("metricas", {})
for _chave in ["score_abnt", "bertscore_f1_medio", "jaccard_medio",
               "jaccard_semantico_medio",    # FastText -- aceita sinonimos
               "similaridade_lexical_media", "score_semantico_hibrido_medio"]:
    _val = _met.get(_chave)
    if _val is not None:
        try:
            _exib = _fmt_num(_val, 3)
        except Exception:
            _exib = str(_val)
        print(f"  {_chave:<40}: {_exib}")

# ── 7. Relatório textual ──────────────────────────────────────────────────────
print("\n── Relatório interpretativo ─────────────────────────────────────────────────")
_rel = saida.get("relatorio_textual", "")
if _rel:
    display(Markdown(_rel))
else:
    print("  (Relatório não disponível)")

## 13. Relatório interpretativo com LLM via Hugging Face

Esta etapa mantém duas saídas complementares:

- **relatório interpretativo heurístico**, gerado a partir dos indicadores estruturados do pipeline
- **relatório interpretativo com LLM via Hugging Face**, usando Llama 3.2 quando o acesso ao modelo estiver habilitado

Modelo padrão desta etapa: `meta-llama/Llama-3.2-1B-Instruct`.

**Pré-requisitos para a LLM:**

- aceitar a licença do modelo na Hugging Face
- disponibilizar `HF_TOKEN`, `HUGGINGFACEHUB_API_TOKEN` ou `HUGGINGFACE_API_KEY` no ambiente do kernel ou em `.env`
- opcionalmente usar o script `setup_llm_env.py` para checar dependências e orientar a configuração do token

Se o token não estiver disponível, a célula informa a indisponibilidade e mantém a saída heurística como fallback.

In [ ]:
from IPython.display import Markdown, display

import os
from pathlib import Path

import torch
from dotenv import load_dotenv
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

USAR_LLM = False  # deixe True apenas quando quiser carregar e executar a LLM

# chamar_llm_analise_abnt (importada do notebook de funcoes via %run) usa
# a Hugging Face Inference API (hospedada) como primario, com fallback para
# Ollama local. O modelo vem de MODEL_NAME_LLM_HF do notebook de funcoes
# (meta-llama/Llama-3.1-8B-Instruct desde 2026-07 -- o Mistral-7B deixou de
# ser servido como chat pela Inference API). NAO redefinimos o nome aqui,
# para nao sombrear o valor do notebook de funcoes.
#
# A implementacao abaixo (gerar_relatorio_llm_local) carrega o modelo
# localmente via transformers -- alternativa mais pesada, mantida para
# compatibilidade e testes offline.
TOKEN_ENV_VARS = ("HF_TOKEN", "HUGGINGFACEHUB_API_TOKEN", "HUGGINGFACE_API_KEY")

def carregar_dotenv_local():
    candidatos = [Path.cwd() / ".env", Path.cwd() / "PLN_PROJETO_FINAL" / ".env"]
    for candidato in candidatos:
        if candidato.exists():
            load_dotenv(candidato, override=False)
            return candidato
    return None

def ler_token_hf():
    for nome in TOKEN_ENV_VARS:
        valor = os.environ.get(nome)
        if valor:
            if nome != "HF_TOKEN" and not os.environ.get("HF_TOKEN"):
                os.environ["HF_TOKEN"] = valor
            return nome, valor
    return None, None

DOTENV_CARREGADO = carregar_dotenv_local()
HF_TOKEN_ENV_NAME, HF_TOKEN = ler_token_hf()
_HF_LLM_PIPE = None
_HF_LLM_PIPE_MODEL = None

def montar_prompt_relatorio_curto(indicadores: dict) -> str:
    return f"""
Você é um especialista em análise estrutural de artigos científicos, ABNT/NBR 6022 e PLN.

Gere um relatório interpretativo curto com base apenas nestes indicadores:
{indicadores}

Regras:
- Não invente informações.
- Não faça avaliação oficial da ABNT.
- O score é heurístico.
- Dê sugestões práticas.
- Se houver seções com 'nivel': 'observacao', explique que o conteúdo existe mas falta o título.
- Se houver 'nivel': 'critico', destaque como problema grave.

Organize em:
1. Diagnóstico geral
2. Problemas encontrados
3. Sugestões de melhoria
"""

def _carregar_pipeline_llm_hf(modelo: str = MODEL_NAME_LLM_HF):
    global _HF_LLM_PIPE, _HF_LLM_PIPE_MODEL

    if _HF_LLM_PIPE is not None and _HF_LLM_PIPE_MODEL == modelo:
        return _HF_LLM_PIPE

    if not HF_TOKEN:
        raise RuntimeError(
            "Token da Hugging Face não encontrado. Defina HF_TOKEN, HUGGINGFACEHUB_API_TOKEN ou HUGGINGFACE_API_KEY no ambiente do kernel ou em .env."
        )

    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    tokenizer_hf = AutoTokenizer.from_pretrained(modelo, token=HF_TOKEN)
    model_hf = AutoModelForCausalLM.from_pretrained(
        modelo,
        token=HF_TOKEN,
        torch_dtype=dtype,
        device_map="auto" if torch.cuda.is_available() else None,
    )
    _HF_LLM_PIPE = pipeline(
        "text-generation",
        model=model_hf,
        tokenizer=tokenizer_hf,
    )
    _HF_LLM_PIPE_MODEL = modelo
    return _HF_LLM_PIPE

def gerar_relatorio_llm_local(indicadores: dict, modelo: str = MODEL_NAME_LLM_HF) -> str:
    """Gera relatorio interpretativo carregando MODEL_NAME_LLM_HF localmente, com fallback explicativo."""
    prompt = montar_prompt_relatorio_curto(indicadores)

    if not isinstance(indicadores, dict) or not indicadores:
        return "**LLM indisponível para geração:** indicadores vazios ou inválidos."

    if not USAR_LLM:
        return (
            "**LLM desativada nesta execução.**\n\n"
            "A etapa de LLM está preparada, mas não é carregada por padrão. "
            "Para testar a LLM local/Hugging Face, defina `USAR_LLM = True` e garanta token/modelo disponível."
        )

    if not HF_TOKEN:
        return (
            f"**LLM Hugging Face planejada:** `{modelo}`\n\n"
            "Nenhum token da Hugging Face foi encontrado nesta sessão.\n"
            "Defina `HF_TOKEN`, `HUGGINGFACEHUB_API_TOKEN` ou `HUGGINGFACE_API_KEY` em `.env` ou no ambiente, aceite a licença do modelo e execute novamente para ativar esta etapa."
        )

    try:
        pipe = _carregar_pipeline_llm_hf(modelo)
        mensagens = [
            {
                "role": "system",
                "content": (
                    "Você é um assistente técnico especialista em ABNT/NBR 6022, "
                    "estrutura de artigos científicos e processamento de linguagem natural."
                ),
            },
            {
                "role": "user",
                "content": prompt,
            },
        ]

        if hasattr(pipe.tokenizer, "apply_chat_template"):
            prompt_final = pipe.tokenizer.apply_chat_template(
                mensagens,
                tokenize=False,
                add_generation_prompt=True,
            )
        else:
            prompt_final = (
                "Sistema: Você é um assistente técnico especialista em ABNT/NBR 6022, "
                "estrutura de artigos científicos e PLN.\n\n"
                f"Usuário: {prompt}\n\nAssistente:"
            )

        resposta = pipe(
            prompt_final,
            max_new_tokens=350,
            do_sample=False,
            return_full_text=False,
        )
        conteudo = resposta[0].get("generated_text", "") if resposta else ""
        if conteudo and str(conteudo).strip():
            return str(conteudo).strip()
        return (
            f"**LLM Hugging Face executada com saída vazia:** `{modelo}`\n\n"
            "O modelo respondeu sem conteúdo textual útil."
        )
    except Exception as e:
        return (
            f"**Falha ao gerar relatório com LLM Hugging Face (`{modelo}`):** {e}\n\n"
            "Mantenha o relatório interpretativo heurístico como fallback e verifique token, licença do modelo e memória disponível."
        )

montar_prompt_feedback_curto = montar_prompt_relatorio_curto
gerar_feedback_llm_local = gerar_relatorio_llm_local

In [ ]:
# Relatório interpretativo baseado na saída estruturada do pipeline
# Usa um artigo já processado/amostrado, mas evita montar indicadores manualmente incompletos.

if df_resultados_pipeline.empty:
    raise ValueError("df_resultados_pipeline está vazio. Execute a seção 11 antes desta célula.")

pos_artigo = min(4, len(df_resultados_pipeline) - 1)
idx_artigo = df_resultados_pipeline['indice_df'].iloc[pos_artigo]
texto_artigo = df_pt.loc[idx_artigo, coluna_texto]

out_real = analisar_artigo(
    texto_artigo,
    nlp=nlp,
    tokenizer=tokenizer,
    model=model,
    stopwords_pt=STOPWORDS_LEXICO,
    ft_model=ft_model,
    kw_model=kw_model,
    ner_pipeline_lenerbr=ner_pipeline_lenerbr,
    ner_pipeline_scierc=ner_pipeline_scierc,
    zs_pipeline=zs_pipeline,
)
saida_real = gerar_saida_analise(out_real)

# Palavras-chave para enriquecer o relatório heurístico
palavras_chave_real = []
for dados_secao in out_real.get("_lexico", {}).get("por_secao", {}).values():
    for termo, _score in dados_secao.get("tfidf", [])[:5]:
        if termo not in palavras_chave_real:
            palavras_chave_real.append(termo)
palavras_chave_real = palavras_chave_real[:8]

relatorio_heuristico = gerar_feedback_interpretativo(
    out_real.get("_indicadores", {}),
    palavras_chave=palavras_chave_real,
)

display(Markdown("## Relatório interpretativo heurístico\n\n" + relatorio_heuristico))

if USAR_LLM:
    relatorio_llm_local = gerar_relatorio_llm_local(
        out_real.get("_indicadores", {}),
        modelo=MODEL_NAME_LLM_HF,
    )
    display(Markdown("## Relatório interpretativo com LLM Hugging Face\n\n" + relatorio_llm_local))
else:
    display(Markdown(
        "## Relatório interpretativo com LLM Hugging Face\n\n"
        "LLM desativada nesta execução (`USAR_LLM = False`). "
        "Use o relatório heurístico acima como saída principal por enquanto."
    ))

# Tabela compacta das seções obrigatórias
sec_rows = []
for secao, dados in saida_real.get("secoes_obrigatorias", {}).items():
    sec_rows.append({
        "secao": secao,
        "presente": dados.get("presente"),
        "status": dados.get("status"),
        "score_lexical": dados.get("score_lexical"),
        "score_semantico": dados.get("score_semantico"),
        "score_final": dados.get("score_final"),
        "jaccard_score": dados.get("jaccard_score"),
        "indicador_lexical": dados.get("indicador_lexical"),
    })
display(pd.DataFrame(sec_rows))


## 14. Protótipo para análise de novo arquivo



Entrada suportada nesta versão:



- `.pdf` com extração via PyMuPDF (`fitz`)

- `.docx` com extração via `python-docx`



Fluxo desta etapa:



1. extrair texto do arquivo

2. executar `analisar_artigo`

3. gerar saída estruturada

4. gerar relatório heurístico

5. tentar gerar relatório com Llama 3.2 via Hugging Face

Limites desta versao:

- PDFs escaneados ainda exigem OCR, tratado como melhoria futura

- nao ha RAG nesta etapa; o documento e analisado diretamente, sem indexacao vetorial ou recuperacao contextual

- OCR + RAG ficam como extensoes futuras para ampliar cobertura de entrada e suporte contextual


In [ ]:
caminho_arquivo_pdf_exemplo = r"c:\Users\Maria Beatriz\Downloads\rsl_PLN.pdf"

caminho_arquivo_docx_exemplo = r"c:\Users\Maria Beatriz\Downloads\Artigo ProDuc.docx"

caminho_arquivo = caminho_arquivo_docx_exemplo  # troque para o PDF se quiser testar o outro fluxo



texto_arquivo = extrair_texto_arquivo(caminho_arquivo)

if not texto_arquivo or not str(texto_arquivo).strip():

    raise ValueError(

        f"Nenhum texto foi extraído do arquivo informado: {caminho_arquivo}. "

        "Se o PDF for apenas imagem ou estiver corrompido, teste um .docx ou um PDF com camada de texto."

    )



out_arquivo = analisar_artigo(

    texto_arquivo,

    nlp=nlp,

    tokenizer=tokenizer,

    model=model,

    stopwords_pt=STOPWORDS_LEXICO,

    ft_model=ft_model,

    kw_model=kw_model,

    ner_pipeline_lenerbr=ner_pipeline_lenerbr,

    ner_pipeline_scierc=ner_pipeline_scierc,

    zs_pipeline=zs_pipeline,

)

saida_arquivo = gerar_saida_analise(out_arquivo)



palavras_chave_arquivo = []

for dados_secao in out_arquivo.get("_lexico", {}).get("por_secao", {}).values():

    for termo, _score in dados_secao.get("tfidf", [])[:5]:

        if termo not in palavras_chave_arquivo:

            palavras_chave_arquivo.append(termo)

palavras_chave_arquivo = palavras_chave_arquivo[:8]



display(Markdown("## Saída estruturada do arquivo\n\n"))

print(f"Arquivo analisado: {caminho_arquivo}")

print(f"Score geral: {saida_arquivo.get('score_geral')} | Status: {saida_arquivo.get('status_geral')}")

print(f"Idioma: {saida_arquivo.get('idioma')} | Seções presentes: {saida_arquivo.get('qtd_secoes_presentes')}")

print(f"Palavras-chave extraídas para o relatório: {palavras_chave_arquivo}")



feedback_arquivo = gerar_feedback_interpretativo(
    out_arquivo.get("_indicadores", {}),
    palavras_chave=palavras_chave_arquivo,
)
display(Markdown("## Relatório heurístico do arquivo\n\n" + feedback_arquivo))

if USAR_LLM:
    relatorio_llm_arquivo = gerar_feedback_llm_local(
        out_arquivo.get("_indicadores", {}),
        modelo=MODEL_NAME_LLM_HF,
    )
    display(Markdown("## Relatório com LLM Hugging Face\n\n" + relatorio_llm_arquivo))
else:
    display(Markdown(
        "## Relatório com LLM Hugging Face\n\n"
        "LLM desativada nesta execução (`USAR_LLM = False`). "
        "Ative apenas quando quiser testar a etapa de geração textual com modelo."
    ))


## 15. Conclusões

- O pipeline usa `df_pt` para análises finais.
- O score ABNT é uma métrica heurística, **não** uma validação oficial da norma.
- A validação semântica híbrida (heurística + BERTimbau) diferencia título ausente de conteúdo ausente.
- Coerência semântica via BERTimbau é computacionalmente custosa — use em amostras menores.
- BERTimbau e a LLM via Hugging Face são módulos complementares, não substitutos da avaliação humana.
- OCR para PDFs escaneados permanece como melhoria futura.
- RAG permanece como melhoria futura para cenários que exijam memoria externa, recuperacao contextual ou apoio a bases auxiliares.


In [ ]:
# Exporta relatório em Markdown
from pathlib import Path

report_dir = Path(base_dir_coleta_scielo) / 'resultados_pipeline' / 'relatorio_v3'
report_dir.mkdir(parents=True, exist_ok=True)
fig_paths  = {}

# 1) Filtro de idioma
if 'df_filtrado' in globals() and 'manter_dataset_pt' in df_filtrado.columns:
    plt.figure(figsize=(6, 4))
    vc = df_filtrado['manter_dataset_pt'].value_counts().rename(index={True:'PT', False:'Não PT'})
    sns.barplot(x=vc.index, y=vc.values)
    plt.title('Distribuição após filtro de idioma')
    plt.xlabel('Classe'); plt.ylabel('Quantidade de artigos')
    p = report_dir / 'grafico_01_filtro_idioma.png'
    plt.savefig(p, dpi=110); plt.close()
    fig_paths['filtro'] = p

# 2) Histograma seções
if 'df_sec' in globals() and 'n_secoes_detectadas' in df_sec.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(df_sec['n_secoes_detectadas'].dropna(), bins=12, kde=True)
    plt.title('Distribuição do número de seções detectadas')
    plt.xlabel('Número de seções'); plt.ylabel('Artigos')
    p = report_dir / 'grafico_02_secoes.png'
    plt.savefig(p, dpi=110); plt.close()
    fig_paths['secoes'] = p

# 3) BoW/TF-IDF
if 'df_top_bow' in globals() and 'df_top_tfidf' in globals():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.barplot(data=df_top_bow.head(15),   y='termo', x='frequencia', ax=axes[0])
    axes[0].set_title('Top termos BoW')
    sns.barplot(data=df_top_tfidf.head(15), y='termo', x='score',      ax=axes[1])
    axes[1].set_title('Top termos TF-IDF')
    p = report_dir / 'grafico_03_bow_tfidf.png'
    plt.savefig(p, dpi=110); plt.close()
    fig_paths['bow_tfidf'] = p

# Exporta CSVs
for nome, var in [
    ("saida_df_sec_head20.csv",      df_sec.head(20)           if 'df_sec' in globals() else None),
    ("saida_top_bow.csv",            df_top_bow                if 'df_top_bow' in globals() else None),
    ("saida_top_tfidf.csv",          df_top_tfidf              if 'df_top_tfidf' in globals() else None),
    ("saida_df_resultados_pipeline.csv", df_resultados_pipeline if 'df_resultados_pipeline' in globals() else None),
]:
    if var is not None:
        var.to_csv(report_dir / nome, index=False, encoding='utf-8-sig')

# Monta markdown
linhas = ["# Relatório de Resultados — Pipeline ABNT v3", "",
          "Gerado automaticamente a partir de `analise_pipeline_abnt_apresentacao_v3_1.ipynb`.", ""]

if 'df' in globals():       linhas.append(f"- Total de artigos no dataset: {len(df)}")
if 'df_pt' in globals():    linhas.append(f"- Artigos mantidos (PT): {len(df_pt)}")
if 'df_nao_pt' in globals(): linhas.append(f"- Artigos fora do critério PT: {len(df_nao_pt)}")
linhas.extend(["", "## Gráficos", ""])

for k, titulo in [('filtro','Filtro de idioma'),('secoes','Seções detectadas'),('bow_tfidf','BoW/TF-IDF')]:
    if k in fig_paths:
        rel = './' + str(fig_paths[k].name)
        linhas += [f"### {titulo}", f"![{titulo}]({rel})", ""]

linhas += ["## Saídas tabulares", "",
           "- saida_df_sec_head20.csv", "- saida_top_bow.csv",
           "- saida_top_tfidf.csv", "- saida_df_resultados_pipeline.csv", ""]

if 'relatorio_heuristico' in globals():
    linhas += ["## Feedback interpretativo (exemplo)", "", relatorio_heuristico]

md_path = report_dir / 'RELATORIO_v3_1.md'
md_path.write_text('\n'.join(linhas), encoding='utf-8')
print(f"Relatório salvo em: {md_path}")


## 16. Persistencia local - JSON + Parquet

Salva os resultados do pipeline em arquivos locais para analise posterior, sem depender de schema relacional.

A persistencia passa a usar:

- um arquivo JSON completo por artigo
- um arquivo Parquet com o resumo tabular consolidado
- uma pasta local dentro de `resultados_pipeline`


In [ ]:
import json
from datetime import datetime
from pathlib import Path

BASE_RESULTADOS = Path(base_dir_coleta_scielo) / "resultados_pipeline"
DIR_JSON_RESULTADOS = BASE_RESULTADOS / "resultados_json"
PARQUET_PATH = BASE_RESULTADOS / "resultados_abnt_resumo.parquet"

DIR_JSON_RESULTADOS.mkdir(parents=True, exist_ok=True)


def _serializar_json_local(obj):
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    if isinstance(obj, set):
        return sorted(obj)
    return str(obj)


def montar_linha_resumo(nome_artigo: str, resultado: dict) -> dict:
    """Gera a linha tabular resumida a partir da saida estruturada do pipeline."""
    saida = gerar_saida_analise(resultado)
    ind = resultado.get("_indicadores", {})
    cit = resultado.get("citacoes", {})
    secoes_detectadas = ", ".join(
        k for k in resultado.get("_secoes_raw", {})
        if not k.startswith("_")
    )

    return {
        "nome_artigo": nome_artigo,
        "data_analise": datetime.now().isoformat(timespec="seconds"),
        "score_geral": saida.get("score_geral"),
        "status_geral": saida.get("status_geral"),
        "idioma": saida.get("idioma", ""),
        "status_idioma": saida.get("status_idioma", ""),
        "score_abnt": ind.get("score_abnt_heuristico"),
        "conforme_nbr6022": bool(ind.get("conforme_nbr6022", False)),
        "n_criticos": ind.get("n_criticos", 0),
        "n_avisos": ind.get("n_avisos", 0),
        "n_observacoes": ind.get("n_observacoes", 0),
        "citacoes_diretas": cit.get("diretas", {}).get("count", 0),
        "citacoes_indiretas": cit.get("indiretas", {}).get("count", 0),
        "status_citacoes": cit.get("status", ""),
        "score_citacoes": cit.get("score_citacoes", 0.0),
        "qtd_secoes_presentes": saida.get("qtd_secoes_presentes", 0),
        "secoes_detectadas": secoes_detectadas,
    }


def salvar_resultado_json_local(nome_artigo: str, resultado: dict, diretorio_json: Path) -> Path:
    """Salva a saida completa do artigo em JSON local."""
    caminho_json = diretorio_json / f"{nome_artigo}.json"
    payload = {
        "nome_artigo": nome_artigo,
        "data_analise": datetime.now().isoformat(timespec="seconds"),
        "resultado": gerar_saida_analise(resultado),
        "resultado_bruto": resultado,
    }
    caminho_json.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2, default=_serializar_json_local),
        encoding="utf-8",
    )
    return caminho_json


def salvar_resumo_parquet(df_resumo: pd.DataFrame, caminho_parquet: Path) -> Path:
    """Salva o resumo tabular em Parquet local."""
    try:
        df_resumo.to_parquet(caminho_parquet, index=False)
    except ImportError as e:
        raise RuntimeError(
            "Parquet requer 'pyarrow' ou 'fastparquet' no ambiente atual."
        ) from e
    return caminho_parquet


print(f"Diretorio JSON: {DIR_JSON_RESULTADOS}")
print(f"Arquivo Parquet: {PARQUET_PATH}")

_variaveis_persistencia_ok = all(
    v in globals() for v in ["df_resultados_pipeline", "df_pt", "coluna_texto"]
)

if _variaveis_persistencia_ok:
    linhas_resumo = []
    caminhos_json = []

    for _, linha_pipeline in df_resultados_pipeline.iterrows():
        idx_art = int(linha_pipeline["indice_df"])
        nome_artigo = f"artigo_{idx_art}"
        texto_art = str(df_pt.loc[idx_art, coluna_texto])
        resultado_art = analisar_artigo(
            texto_art,
            nlp=nlp, tokenizer=tokenizer, model=model,
            stopwords_pt=STOPWORDS_LEXICO,
            ft_model=ft_model, kw_model=kw_model,
            ner_pipeline_lenerbr=ner_pipeline_lenerbr,
            ner_pipeline_scierc=ner_pipeline_scierc,
            zs_pipeline=zs_pipeline,
        )

        caminho_json = salvar_resultado_json_local(nome_artigo, resultado_art, DIR_JSON_RESULTADOS)
        caminhos_json.append(str(caminho_json))
        linhas_resumo.append(montar_linha_resumo(nome_artigo, resultado_art))
        print(f"  Salvo {nome_artigo} -> {caminho_json.name}")

    df_resumo_local = pd.DataFrame(linhas_resumo)
    salvar_resumo_parquet(df_resumo_local, PARQUET_PATH)

    print(f"\n{len(caminhos_json)} artigo(s) salvos em JSON.")
    print(f"Resumo tabular salvo em: {PARQUET_PATH}")
    display(df_resumo_local)
else:
    print("Aviso: execute a celula do pipeline (secao 11) antes de salvar os resultados.")

In [ ]:
# Leitura e visualizacao dos resultados salvos em JSON + Parquet
from pathlib import Path

BASE_RESULTADOS = Path(base_dir_coleta_scielo) / "resultados_pipeline"
DIR_JSON_RESULTADOS = BASE_RESULTADOS / "resultados_json"
PARQUET_PATH = BASE_RESULTADOS / "resultados_abnt_resumo.parquet"

if not PARQUET_PATH.exists():
    print(f"Aviso: arquivo Parquet nao encontrado em {PARQUET_PATH}")
elif not DIR_JSON_RESULTADOS.exists():
    print(f"Aviso: diretorio JSON nao encontrado em {DIR_JSON_RESULTADOS}")
else:
    df_banco = pd.read_parquet(PARQUET_PATH)
    arquivos_json = sorted(DIR_JSON_RESULTADOS.glob("*.json"))

    print(f"Registros no Parquet: {len(df_banco)}")
    print(f"Arquivos JSON locais: {len(arquivos_json)}")
    display(df_banco)

    if arquivos_json:
        print("\nExemplos de JSON salvos:")
        for caminho in arquivos_json[:5]:
            print(f"- {caminho.name}")

    if not df_banco.empty:
        plt.figure(figsize=(10, 4))
        cores_status = {
            "Em conformidade": "#2ca02c",
            "Necessita revisão": "#ff7f0e",
            "Necessita melhorias": "#1f77b4",
            "Fora de conformidade": "#d62728",
        }
        cores = df_banco["status_geral"].map(cores_status).fillna("#aec7e8")
        plt.bar(df_banco["nome_artigo"], df_banco["score_geral"], color=cores)
        plt.axhline(85, color="green", linestyle="--", alpha=0.6, label="Em conformidade >= 85")
        plt.axhline(70, color="orange", linestyle="--", alpha=0.6, label="Revisao >= 70")
        plt.title("Score ABNT por artigo (persistencia local)")
        plt.xlabel("Artigo")
        plt.ylabel("Score (0-100)")
        plt.xticks(rotation=20, ha="right")
        plt.legend(fontsize=8)
        plt.tight_layout()
        plt.show()


## 17. Avaliação quantitativa da detecção de seções (acurácia, precisão, recall e F1)

As métricas comparam a **presença predita** de cada seção (camadas 1-6 da detecção
estrutural) com um **gabarito revisado manualmente** — sem gabarito humano não há
como calcular métricas supervisionadas (por isso `precision/recall/f1/accuracy`
ficavam como `None` no dicionário de métricas do pipeline).

Fluxo em 3 passos:

1. **Gerar o template** (primeira célula abaixo): amostra 30 artigos e salva
   `gabarito_secoes_template.csv` com a presença predita (0/1) por seção.
2. **Revisar manualmente**: copie o arquivo para `gabarito_secoes.csv` e corrija
   os 0/1 (1 = a seção realmente existe no artigo). O template original **não deve
   ser editado** — ele guarda as predições (`y_pred`); a cópia corrigida é o `y_true`.
   Para conferir um artigo no notebook: `print(df_pt.loc[INDICE, coluna_texto][:3000])`.
3. **Calcular** (segunda célula): acurácia/precisão/recall/F1 por seção +
   agregados micro e macro, com gráfico de F1 por seção.

A unidade de avaliação é a decisão binária "seção presente?" por (artigo, seção).


In [ ]:
# Passo 1 -- gera a planilha-gabarito com as predicoes do pipeline.
# Revisao manual: copie o template para gabarito_secoes.csv e corrija os 0/1.
# NAO edite o template em si: ele guarda as predicoes originais (y_pred).
import os

CAMINHO_GABARITO_TEMPLATE = "gabarito_secoes_template.csv"
CAMINHO_GABARITO_REVISADO = "gabarito_secoes.csv"
N_AMOSTRA_GABARITO = 30

if os.path.isfile(CAMINHO_GABARITO_TEMPLATE):
    print(f"Template ja existe: {CAMINHO_GABARITO_TEMPLATE}")
    print("Apague o arquivo se quiser gerar de novo (as predicoes serao recalculadas).")
else:
    amostra_gab = df_pt.dropna(subset=[coluna_texto]).sample(
        n=min(N_AMOSTRA_GABARITO, len(df_pt)), random_state=42
    )
    linhas_gab = []
    for i, (idx, row) in enumerate(amostra_gab.iterrows(), start=1):
        print(f"Predizendo secoes {i}/{len(amostra_gab)} -- indice {idx}")
        texto_gab = preparar_texto_para_estrutura(str(row[coluna_texto]))
        sec_gab = detectar_secoes(texto_gab, nlp=nlp, tokenizer=tokenizer,
                                  model=model, zs_pipeline=zs_pipeline)
        linha = {"indice_df": idx}
        for s in ORDEM_NBR6022:
            linha[s] = int(s in sec_gab and isinstance(sec_gab.get(s), int))
        linhas_gab.append(linha)
    pd.DataFrame(linhas_gab).to_csv(CAMINHO_GABARITO_TEMPLATE, index=False, encoding="utf-8-sig")
    print()
    print(f"Template salvo: {CAMINHO_GABARITO_TEMPLATE}")
    print(f"Proximo passo: copie para {CAMINHO_GABARITO_REVISADO}, corrija os 0/1")
    print("(1 = a secao existe de fato no artigo) e rode a celula seguinte.")


In [ ]:
# Passo 3 -- metricas: y_true = gabarito revisado, y_pred = template (predicoes).
# Avaliacao binaria de PRESENCA por (artigo, secao).
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

if not (os.path.isfile(CAMINHO_GABARITO_TEMPLATE) and os.path.isfile(CAMINHO_GABARITO_REVISADO)):
    print("Faltam arquivos para avaliar:")
    print(f"  template (predicoes) : {CAMINHO_GABARITO_TEMPLATE} -> existe? {os.path.isfile(CAMINHO_GABARITO_TEMPLATE)}")
    print(f"  gabarito (revisado)  : {CAMINHO_GABARITO_REVISADO} -> existe? {os.path.isfile(CAMINHO_GABARITO_REVISADO)}")
    print("Rode a celula anterior e revise o gabarito antes desta.")
else:
    df_pred = pd.read_csv(CAMINHO_GABARITO_TEMPLATE, encoding="utf-8-sig").set_index("indice_df").sort_index()
    df_true = pd.read_csv(CAMINHO_GABARITO_REVISADO, encoding="utf-8-sig").set_index("indice_df").sort_index()

    # Alinha pela intersecao: o gabarito revisado pode ter EXCLUIDO documentos
    # que nao sao artigos (erratas, resenhas, listas) -- decisao de anotacao.
    comuns = df_pred.index.intersection(df_true.index)
    if len(comuns) == 0:
        raise ValueError("Nenhum indice em comum entre template e gabarito -- confira os arquivos.")
    if len(comuns) < len(df_pred):
        print(f"Avaliando {len(comuns)} artigos (excluidos do gabarito: {sorted(set(df_pred.index) - set(comuns))})")
    df_pred = df_pred.loc[comuns]
    df_true = df_true.loc[comuns]

    secoes_aval = [s for s in ORDEM_NBR6022 if s in df_pred.columns and s in df_true.columns]

    linhas_met, y_true_all, y_pred_all = [], [], []
    for s in secoes_aval:
        y_true = df_true[s].astype(int).tolist()
        y_pred = df_pred[s].astype(int).tolist()
        linhas_met.append({
            "secao": s,
            "positivos_reais": int(sum(y_true)),
            "acuracia": accuracy_score(y_true, y_pred),
            "precisao": precision_score(y_true, y_pred, zero_division=0),
            "recall":   recall_score(y_true, y_pred, zero_division=0),
            "f1":       f1_score(y_true, y_pred, zero_division=0),
        })
        y_true_all += y_true
        y_pred_all += y_pred

    df_metricas = pd.DataFrame(linhas_met).set_index("secao").round(3)
    display(df_metricas)

    print("Agregado micro (todas as decisoes artigo x secao juntas):")
    print(f"  acuracia = {accuracy_score(y_true_all, y_pred_all):.3f}")
    print(f"  precisao = {precision_score(y_true_all, y_pred_all, zero_division=0):.3f}")
    print(f"  recall   = {recall_score(y_true_all, y_pred_all, zero_division=0):.3f}")
    print(f"  f1       = {f1_score(y_true_all, y_pred_all, zero_division=0):.3f}")
    print("Agregado macro (media simples entre secoes):")
    print(f"  precisao = {df_metricas['precisao'].mean():.3f}")
    print(f"  recall   = {df_metricas['recall'].mean():.3f}")
    print(f"  f1       = {df_metricas['f1'].mean():.3f}")

    plt.figure(figsize=(10, 4))
    df_metricas["f1"].plot(kind="bar", color="#007BFF")
    plt.title("F1 da deteccao de presenca por secao (vs gabarito revisado)")
    plt.ylabel("F1")
    plt.ylim(0, 1.05)
    plt.tight_layout()
    plt.show()


## 18. Comparação do feedback LLM — sem RAG vs com RAG

Gera as duas versões do feedback **para o mesmo artigo inspecionado na seção 11.1**
(par controlado: mesma execução do pipeline, mesma LLM, mesmo prompt-base — a única
diferença é o bloco de trechos normativos recuperados pelo RAG).

- **Sem RAG (baseline)**: a LLM responde só com o texto do artigo + números do pipeline.
- **Com RAG**: consultas derivadas dos problemas detectados recuperam trechos de
  manuais públicos de normalização ABNT (UFC, PUC Minas, UFABC — `PLN_BACKEND/normas_rag/`),
  e a LLM é instruída a fundamentar as recomendações citando a norma (NBR 6022/10520).

Pré-requisito: rodar a seção 11.1 antes (define `resultado`, `idx` e `row`).
Para trocar o artigo comparado, mude o índice na célula da 11.1 e rode as duas de novo.


In [ ]:
# Recuperação RAG para o artigo inspecionado na seção 11.1
import sys
from pathlib import Path

_FRONT = str((Path.cwd().parent.parent / "PLN front-end").resolve())
if _FRONT not in sys.path:
    sys.path.insert(0, _FRONT)
import rag_abnt

if "resultado" not in globals():
    raise ValueError("Execute a seção 11.1 (inspeção manual) antes: ela define `resultado`.")

# texto original do artigo (o prompt usa os primeiros 1500 chars)
texto_llm = str(row[coluna_texto]) if "row" in globals() else resultado["_texto_estruturado"]

indice_rag = rag_abnt.carregar_indice()
if indice_rag is None:
    raise FileNotFoundError(
        "Índice RAG não encontrado em PLN_BACKEND/normas_rag. "
        "Construa uma vez com: rag_abnt.construir_indice(modelo=model)"
    )

consultas_rag = rag_abnt.montar_consultas(resultado)
trechos_rag = rag_abnt.recuperar_trechos(consultas_rag, model, indice_rag, top_k=2)
contexto_rag = rag_abnt.montar_contexto_rag(trechos_rag) or None

print("Consultas derivadas dos problemas detectados pelo pipeline:")
for c in consultas_rag:
    print("  -", c)
print(f"\n{len(trechos_rag)} trechos recuperados:")
for t in trechos_rag:
    print(f"  [{t['score']:.3f}] {t['fonte']}: {t['trecho'][:110]}...")


In [ ]:
# Par controlado: mesma LLM, mesmo resultado -- com e sem o contexto normativo
import re
from IPython.display import Markdown, display

fb_sem_rag = chamar_llm_analise_abnt(texto_llm, resultado)
fb_com_rag = chamar_llm_analise_abnt(texto_llm, resultado, rag_contexto=contexto_rag)


def _bloco(titulo, fb):
    corpo = fb.get("resposta") or f"*(indisponível: {fb.get('status')})*"
    return (f"### {titulo}\n"
            f"*{fb.get('modelo', '?')} via {fb.get('origem', '?')} — status {fb.get('status')}*\n\n{corpo}")


display(Markdown(_bloco("🔹 Feedback SEM RAG (baseline)", fb_sem_rag)))
display(Markdown("---"))
display(Markdown(_bloco("🔸 Feedback COM RAG (fundamentado nos manuais ABNT)", fb_com_rag)))

print()
for nome, fb in (("sem RAG", fb_sem_rag), ("com RAG", fb_com_rag)):
    resp = fb.get("resposta") or ""
    nbrs = re.findall(r"(?i)NBR\s*\d{4,5}", resp)
    print(f"{nome}: {len(resp)} chars | menções a NBR: {len(nbrs)} "
          f"{sorted(set(n.upper().replace(' ', '') for n in nbrs))}")


In [ ]:
# Pares já armazenados pelo site/experimentos (tabela feedback_llm do front-end)
import sqlite3
import pandas as pd

_DB = str(Path(_FRONT) / "base_abnt.db")
try:
    _conn = sqlite3.connect(_DB)
    df_pares_armazenados = pd.read_sql(
        "SELECT artigo_id, rag_ativo, modelo, origem, status, "
        "length(resposta) AS n_chars, data_geracao "
        "FROM feedback_llm ORDER BY artigo_id, data_geracao",
        _conn,
    )
    _conn.close()
    display(df_pares_armazenados)
    print("Texto completo: SELECT resposta FROM feedback_llm WHERE artigo_id=X AND rag_ativo=0 (ou 1)")
except Exception as e:
    print("Banco do site indisponível nesta máquina:", e)
